# **Constraint Programming and Scheduling**

## **1. The mathematical problem**

We want to assign a color to each of four regions while respecting border constraints.

Let the set of regions be:

$$\mathcal{X} = \{A, B, C, D\}.$$

Each region can receive one of three colors. Therefore, for every region $X_i$:

$$D_i = \{red, blue, green\}.$$

The task is to find a complete assignment of colors that satisfies all the constraints. We will first describe this problem mathematically and formally. Only afterwards will we translate the model into Python.

## **2. Which regions are neighbors?**

A **neighbor** is a region that shares a border with another region. Neighboring regions must have different colors. Regions that do not share a border may have the same color; for example, A and D may both be red.

The following image shows the map we are going to model:


<img src="csp_four_region_map_uncolored.svg" alt="Unsolved Map" width="400">

### **2.1 The neighbor relation**

A neighbor is a region that shares a border with another region. Neighboring regions must have different colors. Regions that do not share a border may have the same color; for example, A and D may both be red.

For this map, the neighboring pairs are **unordered pairs**: $\{A,B\}$, $\{A,C\}$, $\{B,C\}$, $\{B,D\}$, and $\{C,D\}$.

Each pair represents one shared border and therefore one constraint: $A \ne B$, $A \ne C$, $B \ne C$, $B \ne D$, and $C \ne D$.

The neighbor relation is symmetric: if A is a neighbor of B, then B is also a neighbor of A. We list each unordered pair only once because $A \ne B$ and $B \ne A$ express the same constraint. In the Python dictionary, however, both directions are stored so that the algorithm can check neighbors from either region.

## **3. Formal model and backtracking algorithm**

A constraint satisfaction problem is described by:

$$CSP = \langle \mathcal{X}, \mathcal{D}, \mathcal{C} \rangle$$

- $\mathcal{X}$ is the set of variables.
- $\mathcal{D}$ is the collection of domains.
- $\mathcal{C}$ is the set of constraints.

For this map, a solution is a complete assignment such as:

$$A=red,\quad B=blue,\quad C=green,\quad D=red.$$

It is valid because every neighboring pair has different colors, while A and D may share red because they are not neighbors.

### **3.1 Backtracking pseudocode**

The algorithm constructs the assignment one variable at a time:

```text
BACKTRACK(assignment):
    if assignment contains every region:
        return assignment

    region <- choose an unassigned region

    for color in DOMAIN(region):
        if color is consistent with assignment:
            add region = color to assignment
            result <- BACKTRACK(assignment)

            if result is not FAILURE:
                return result

            remove region = color from assignment

    return FAILURE
```

The algorithm prunes a branch as soon as a partial assignment violates a constraint. It backtracks when no available color can complete the current partial assignment.

### **3.2 The algorithm's inputs**

Mathematically, the algorithm receives:

- the variables in $\mathcal{X}$;
- the domain $D_i$ for each variable $X_i$;
- the neighbor relation that defines the constraints in $\mathcal{C}$.

The next section translates these mathematical objects into Python. The Python code starts only after the mathematical model and pseudocode are complete.

## **4. Write the problem data in Python**

We now translate the mathematical sets into Python objects, one step at a time. This cell defines data only; it does not search for a solution yet.

- `regions` stores the four variables.
- `colors` stores the domain of each variable.
- `neighboring_regions` stores the constraint relation.

In [10]:
regions = ["A", "B", "C", "D"]

colors = {
    "A": ["red", "blue", "green"],
    "B": ["red", "blue", "green"],
    "C": ["red", "blue", "green"],
    "D": ["red", "blue", "green"],
}

neighboring_regions = {
    "A": ["B", "C"],
    "B": ["A", "C", "D"],
    "C": ["A", "B", "D"],
    "D": ["B", "C"],
}

## **5. Translate the mathematical model into Python**

Now we encode the three mathematical objects:

- `regions` represents the variables in $\mathcal{X}$.
- `colors` represents the domains in $\mathcal{D}$.
- `neighboring_regions` represents the relation used to check the constraints in $\mathcal{C}$.

The next cell implements the generic backtracking solver. It does not know that the variables are map regions; it only uses the three inputs.

In [11]:
def solve_csp(variables, domains, neighbors):
    assignment = {}

    def backtrack():
        if len(assignment) == len(variables):
            return assignment.copy()

        variable = next(variable for variable in variables if variable not in assignment)
        for value in domains[variable]:
            if all(assignment.get(neighbor) != value for neighbor in neighbors[variable]):
                assignment[variable] = value
                result = backtrack()
                if result is not None:
                    return result
                del assignment[variable]
        return None

    return backtrack()

## **6. Run the solver on the map**

The data and the generic solver are now defined. We call the function by passing the three objects that correspond to the mathematical model:

```python
coloring = solve_csp(regions, colors, neighboring_regions)
```

The function returns a dictionary assigning one color to every region.

In [12]:
coloring = solve_csp(regions, colors, neighboring_regions)
print("A valid coloring:", coloring)

A valid coloring: {'A': 'red', 'B': 'blue', 'C': 'green', 'D': 'red'}


<img src="csp_four_region_map_coloring.svg" alt="Solved map" width="400">

## **7. Read the Python implementation step by step**

The call `solve_csp(regions, colors, neighboring_regions)` connects the concrete problem to the generic algorithm:

- `regions` becomes the function's `variables` argument.
- `colors` becomes the `domains` argument.
- `neighboring_regions` becomes the `neighbors` argument.

Inside `solve_csp`, `assignment` starts as an empty dictionary. The nested function `backtrack()` gradually fills it.

1. `len(assignment) == len(variables)` checks whether every region has been colored. If so, the function returns a copy of the complete assignment.
2. `next(...)` selects a region that is still unassigned.
3. `for value in domains[variable]` tries each color allowed for that region.
4. `all(assignment.get(neighbor) != value for neighbor in neighbors[variable])` checks the constraints against already-assigned neighbors.
5. If the color is valid, `assignment[variable] = value` records the decision and `backtrack()` continues recursively.
6. If the recursive call returns `None`, that decision led to a dead end. `del assignment[variable]` undoes it, exactly as in the pseudocode.

The result is a complete assignment such as:

$$A=red,\quad B=blue,\quad C=green,\quad D=red.$$

A and D may have the same color because they are not neighbors.

## **8. Check your understanding**

Use the map example, not an abstract definition, to answer these questions:

1. What are the four variables, and what is the domain of each variable?
2. Why is `{A: red, B: red}` rejected immediately?
3. Why is `{A: red, D: red}` allowed?
4. If A is `red` and B is `blue`, which colors remain possible for C?
5. What does the search return if no color is valid for an unassigned region?

**Instructor prompt:** ask students to write one constraint as an inequality, then explain why that inequality is checked before the assignment is complete.

## **9. Lab: schedule student presentations**

Now apply exactly the same CSP model to a timetable.

Four students (`Ada`, `Ben`, `Cy`, `Dee`) must present in one of three time slots (`9:00`, `10:00`, `11:00`). Students who are working together must present at different times. The conflict pairs are `Ada–Ben`, `Ben–Cy`, and `Cy–Dee`.

### **9.1 Translate the story into AIMA's CSP notation**

- **Variables:** $\mathcal{X} = \{Ada, Ben, Cy, Dee\}$.
- **Domains:** $\mathcal{D} = \{D_{Ada}, D_{Ben}, D_{Cy}, D_{Dee}\}$, where each domain is $\{9:00, 10:00, 11:00\}$.
- **Constraints:** $\mathcal{C}$ contains $Ada \ne Ben$, $Ben \ne Cy$, and $Cy \ne Dee$.

The `conflicts` dictionary will store the neighbor relation. For example, `conflicts["Ben"]` contains Ada and Cy because Ben cannot share a slot with either of them.

A partial assignment such as `{Ada: 9:00, Ben: 9:00}` is invalid because Ada and Ben conflict. The partial assignment `{Ada: 9:00, Dee: 9:00}` is allowed because they are not neighbors. This is exactly the same reasoning as with regions A and D in the map.

### **9.2 Tasks**

1. Read the variables, domains, and conflict relation in the next code cell.
2. Complete `is_consistent` so it returns `False` when an already scheduled neighbor has the same slot.
3. Complete the recursive `search` function. Return a complete assignment, or `None` if every possible choice violates a constraint.
4. Run the supplied checks and add one extra conflict pair of your choice. Does the schedule change?
5. In two sentences, explain where pruning happens and why checking partial assignments is useful.

In [ ]:
students = ["Ada", "Ben", "Cy", "Dee"]
slots = ["9:00", "10:00", "11:00"]
domains = {student: slots[:] for student in students}
conflicts = {
    "Ada": ["Ben"],
    "Ben": ["Ada", "Cy"],
    "Cy": ["Ben", "Dee"],
    "Dee": ["Cy"],
}

def is_consistent(student, slot, assignment):
    # TODO: return False if an already scheduled partner has this slot.
    raise NotImplementedError

def search(assignment):
    # TODO: return a complete solution, or None if all choices fail.
    # Hint: use recursion and remove a tentative assignment after failure.
    raise NotImplementedError

schedule = search({})
print(schedule)

In [ ]:
# Run after completing the starter code.
assert schedule is not None
assert set(schedule) == set(students)
assert all(schedule[a] != schedule[b] for a, partners in conflicts.items() for b in partners)
print("All scheduling constraints are satisfied.")

## **10. Exit ticket and further reading**

Submit your notebook with the completed code, passing checks, the schedule you found, and your short explanation of pruning. In your explanation, name the variables, domains, and at least two constraints from the scheduling problem.

**Extension:** add a fourth slot, or add a preference (such as Ada preferring 10:00) and discuss how optimization differs from finding any feasible schedule. A preference is usually a **soft constraint**: it is desirable, but it may be violated when necessary. A conflict between two students is a **hard constraint**: it must never be violated.

### **AIMA reference**

For the formal definition of constraint satisfaction problems, variables, domains, constraints, backtracking search, and heuristics, read:

- Russell, S. and Norvig, P., *Artificial Intelligence: A Modern Approach*, 4th ed., Chapter 6, **Constraint Satisfaction Problems**.
- In particular, see Sections **6.1 (Defining Constraint Satisfaction Problems)** and **6.3 (Backtracking Search for CSPs)**.

The map-coloring problem in this notebook is the standard graph-coloring CSP discussed in that chapter; the presentation timetable is the same mathematical structure with different variable and domain names.